# META-CXR — Stage-2 Evaluation Notebook (Kaggle, 2×T4 GPU)

Đánh giá pipeline sinh báo cáo X-quang ngực **end-to-end** trên MIMIC-CXR **p10 preprocessed split**:

```text
Image -> BioViL-T / PubMedCLIP -> Q-Former + MHCAC -> 14-abnormality findings -> MedGemma 1.5 -> Generated Report
```

**Metrics output**: BLEU-1/2/3/4, METEOR, ROUGE-L (qua `MIMICEvalCap` có sẵn trong repo), BERTScore + CheXpert-style classification F1 cho 14 abnormalities.

---

## Resources & Kaggle datasets

Attach thủ công qua **Notebook Settings -> Add Data**:

1. **mimic-cxr-jpg-lite** — chest X-ray JPG files + raw metadata CSVs:
   `mimic-cxr-2.0.0-chexpert.csv`, `mimic-cxr-2.0.0-metadata.csv`
2. **mimic-cxr-p10-processed** — split sau preprocessing:
   `p10_train.csv`, `p10_val.csv`, `p10_test.csv`, `all_data.csv`
   (notebook cũng chấp nhận alias `train.csv`, `val.csv`, `test.csv`)
3. **meta-cxr-checkpoints** — Q-Former + MHCAC stage-1 weights, file `.pth`
4. *(Optional)* **google/medgemma-1.5-4b-it** — nếu đã tải model thành Kaggle Dataset, notebook sẽ ưu tiên dùng bản mounted; nếu không, Cell 7 sẽ tải từ Hugging Face.

Không cần attach dataset `mimic-cxr-reported` cho notebook eval này vì ground truth findings được đọc trực tiếp từ CSV đã preprocessing.

### Kaggle environment

- Accelerator: GPU **T4 × 2**
- Internet: **ON** nếu cần tải dependency / MedGemma / pretrained encoder
- Hugging Face: cần accept Health AI Developer Foundations terms của `google/medgemma-1.5-4b-it`; nếu model bị gated, thêm Kaggle Secret `HF_TOKEN` hoặc `HUGGINGFACE_TOKEN`.

---

## Tunable parameters (Cell 5)

`EVAL_LIMIT`, `EVAL_SPLIT`, `BATCH_SIZE_IMG`, `NUM_BEAMS`, `MAX_NEW_TOKENS`, `REPORT_LANGUAGE`, `RUN_BERTSCORE`, `RUN_CHEXPERT_F1`, `TRY_TEXT_LABELER`.

## Expected runtime (2×T4)

| Mode | Samples | Time |
|------|---------|------|
| Smoke test | 4 | ~1-2 phút |
| p10 test split | ~2,000 | phụ thuộc MedGemma generation, thường nhiều giờ hơn Vicuna |
| p10 train split | ~18,000 | nhiều giờ, chỉ nên chạy khi thật cần |

> Bắt đầu với `EVAL_LIMIT = 4` để smoke test đường dẫn dataset/checkpoint/model trước khi chạy full split.

---


In [ ]:
# Cell 1 — Install dependencies for META-CXR + MedGemma 1.5
import subprocess, sys, os

PIP_INSTALL = [
    "transformers>=4.50.0",
    "accelerate>=0.26.0",
    "timm>=0.9.0",
    "loralib==0.1.1",
    "omegaconf==2.3.0",
    "nltk>=3.9",
    "pycocoevalcap",
    "hi-ml-multimodal",
    "huggingface_hub>=0.25.0",
    "sentencepiece",
    "scikit-image",
    "scikit-learn",
    "bert-score",
    "wandb",
]
for pkg in PIP_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# --- Detect Java for METEOR (and optional CheXpert labeler) ---
def _detect_java_home():
    try:
        which = subprocess.check_output(["which", "java"]).decode().strip()
        real = subprocess.check_output(["readlink", "-f", which]).decode().strip()
        return real.replace("/bin/java", "")
    except subprocess.CalledProcessError:
        return "/usr/lib/jvm/java-8-openjdk-amd64/jre"

JAVA_HOME = _detect_java_home()
JAVA_PATH = f"{JAVA_HOME}/bin:"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_PATH + os.environ.get("PATH", "")
print(f"JAVA_HOME = {JAVA_HOME}")

# --- NLTK data for METEOR ---
import nltk
for resource in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"NLTK download {resource!r} failed: {e}")
print("Dependencies installed.")


In [ ]:
# Cell 2 — Kaggle credentials + optional Hugging Face / W&B tokens
import os, json

HAVE_KAGGLE_CREDS = False
KAGGLE_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    KAGGLE_USER = secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = secrets.get_secret("KAGGLE_KEY")
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
        json.dump({"username": KAGGLE_USER, "key": KAGGLE_KEY}, f)
    os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
    HAVE_KAGGLE_CREDS = True
    print(f"Kaggle credentials configured for user {KAGGLE_USER!r}")
except Exception as e:
    print(f"Kaggle credentials not configured ({e!r}) — Cell 11 push-back will be skipped.")

# MedGemma on Hugging Face is gated by Health AI Developer Foundations terms.
# Add Kaggle Secret HF_TOKEN (or HUGGINGFACE_TOKEN) after accepting the model terms.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            hf_token = secrets.get_secret(secret_name)
            if hf_token:
                break
        except Exception:
            pass
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
        print("Hugging Face token loaded from Kaggle Secrets.")
    else:
        print("No HF_TOKEN/HUGGINGFACE_TOKEN secret found; public or cached HF access will be used.")
except Exception as e:
    print(f"Hugging Face token not configured ({e!r}); set HF_TOKEN if MedGemma download is denied.")

# Weights & Biases token for logging eval results to project `stage2_meta_cxr`.
# Add Kaggle Secret WANDB_API_KEY (or WANDB_TOKEN) to enable.
HAVE_WANDB_CREDS = False
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_token = None
    for secret_name in ("WANDB_API_KEY", "WANDB_TOKEN"):
        try:
            wandb_token = secrets.get_secret(secret_name)
            if wandb_token:
                break
        except Exception:
            pass
    if wandb_token:
        os.environ["WANDB_API_KEY"] = wandb_token
        HAVE_WANDB_CREDS = True
        print("W&B token loaded from Kaggle Secrets.")
    else:
        print("No WANDB_API_KEY/WANDB_TOKEN secret found; W&B logging will be skipped.")
except Exception as e:
    print(f"W&B token not configured ({e!r}); set WANDB_API_KEY to enable logging.")


In [ ]:
# Cell 3 — Clone META-CXR repo
import os, subprocess, sys

REPO_URL = os.environ.get("REPO_URL", "https://github.com/minhphuong150505/Meta-CXR-Kaggle.git")
REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

head = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"Repo at {REPO_DIR} @ {head}")


In [ ]:
# Cell 4 — Detect & validate mounted Kaggle datasets for p10 preprocessed eval
import os
from pathlib import Path

MOUNT_ROOTS = ["/kaggle/input/datasets", "/kaggle/input"]


def iter_dataset_candidates(max_depth=2):
    """Yield dataset roots mounted either as /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>."""
    for root in MOUNT_ROOTS:
        root_path = Path(root)
        if not root_path.is_dir():
            continue
        stack = [(root_path, 0)]
        while stack:
            path, depth = stack.pop(0)
            yield path
            if depth >= max_depth:
                continue
            try:
                children = sorted([p for p in path.iterdir() if p.is_dir()])
            except OSError as exc:
                print(f"Skip {path}: {exc}")
                continue
            stack.extend((child, depth + 1) for child in children)


def _has_markers(candidate, marker_files):
    for marker in marker_files:
        if (candidate / marker).exists():
            continue
        if not list(candidate.rglob(marker)):
            return False
    return True


def find_dataset(slug_keywords, marker_files):
    """Find first mounted dataset whose path contains a slug keyword and marker files."""
    for candidate in iter_dataset_candidates():
        haystack = str(candidate).lower()
        if not any(kw in haystack for kw in slug_keywords):
            continue
        if _has_markers(candidate, marker_files):
            return str(candidate)
    return None


def pick_file(root, names, required=True):
    root_path = Path(root)
    for name in names:
        direct = root_path / name
        if direct.exists():
            return str(direct)
        matches = sorted(root_path.rglob(name))
        if matches:
            return str(matches[0])
    if required:
        raise FileNotFoundError(f"None of {names} found under {root}")
    return ""


MIMIC_IMG_ROOT = find_dataset(
    ["mimic-cxr-jpg-lite", "mimic-cxr-jpg", "mimic_cxr_jpg"],
    ["mimic-cxr-2.0.0-chexpert.csv"],
)
assert MIMIC_IMG_ROOT, "Mount mimic-cxr-jpg-lite via Notebook Settings -> Add Data"

PROCESSED_ROOT = (
    find_dataset(["mimic-cxr-p10-processed", "mimic_cxr_p10_processed", "p10-processed"], ["p10_test.csv"])
    or find_dataset(["mimic-cxr-p10-processed", "mimic_cxr_p10_processed", "p10-processed"], ["test.csv"])
    or find_dataset(["mimic-cxr-p10-processed", "mimic_cxr_p10_processed", "p10-processed"], ["all_data.csv"])
)
assert PROCESSED_ROOT, "Mount mimic-cxr-p10-processed via Notebook Settings -> Add Data"

PROCESSED_TRAIN_CSV = pick_file(PROCESSED_ROOT, ["p10_train.csv", "train.csv"])
PROCESSED_VAL_CSV = pick_file(PROCESSED_ROOT, ["p10_val.csv", "val.csv"])
PROCESSED_TEST_CSV = pick_file(PROCESSED_ROOT, ["p10_test.csv", "test.csv"])
PROCESSED_ALL_CSV = pick_file(PROCESSED_ROOT, ["p10_all_data.csv", "all_data.csv"], required=False)
PROCESSED_SPLIT_CSV = pick_file(PROCESSED_ROOT, ["mimic-cxr-2.0.0-split-p10.csv"], required=False)

CKPT_ROOT = find_dataset(
    ["meta-cxr-checkpoints", "meta_cxr_checkpoints", "meta-cxr-checkpoint", "mimic_cxr_checkpoint"],
    [],
)
assert CKPT_ROOT, "Mount meta-cxr-checkpoints or the project checkpoint dataset"

# Find the best Q-Former checkpoint: prefer best > last > highest epoch
pth_files = list(Path(CKPT_ROOT).rglob("*.pth"))
assert pth_files, f"No .pth files in {CKPT_ROOT}"


def ckpt_priority(p: Path):
    name = p.name
    if name == "checkpoint_best.pth":
        return (0, 0)
    if name == "checkpoint_last.pth":
        return (1, 0)
    if name.startswith("checkpoint_") and name.endswith(".pth"):
        try:
            return (2, -int(name.replace("checkpoint_", "").replace(".pth", "")))
        except ValueError:
            return (3, 0)
    return (4, 0)


QFORMER_CKPT = str(sorted(pth_files, key=ckpt_priority)[0])

# Optional local MedGemma dataset. If absent, Cell 7 loads google/medgemma-1.5-4b-it from HF.
MEDGEMMA_ROOT = None
for candidate in iter_dataset_candidates(max_depth=3):
    c_lower = str(candidate).lower()
    if "medgemma" in c_lower:
        cfg_candidates = list(Path(candidate).rglob("config.json"))
        if cfg_candidates:
            MEDGEMMA_ROOT = str(cfg_candidates[0].parent)
            break

print("--- Dataset summary ---")
print(f"  MIMIC images       : {MIMIC_IMG_ROOT}")
print(f"  Processed p10 root : {PROCESSED_ROOT}")
print(f"  train split CSV    : {PROCESSED_TRAIN_CSV}")
print(f"  val split CSV      : {PROCESSED_VAL_CSV}")
print(f"  test split CSV     : {PROCESSED_TEST_CSV}")
print(f"  Ckpt picked        : {QFORMER_CKPT}")
print(f"  MedGemma local     : {MEDGEMMA_ROOT or '(none -> will load google/medgemma-1.5-4b-it from Hugging Face)'}")


In [ ]:
# Cell 5 — Tunable evaluation parameters
EVAL_LIMIT       = None       # None = full split; e.g. 4 (smoke), 500 (subset)
EVAL_SPLIT       = "test"     # "train", "val", or "test"
NUM_BEAMS        = 1
MAX_NEW_TOKENS   = 300
BATCH_SIZE_IMG   = 8          # batch size for BLIP image forward (lower to 4 if OOM)
RUN_BERTSCORE    = True       # Standard report-generation semantic metric used in the paper
RUN_CHEXPERT_F1  = True       # classification F1 over 14 abnormalities from Q-Former logits
TRY_TEXT_LABELER = False      # Java CheXpert labeler on generated text (fragile, off by default)
EXPLAIN_OUTPUT   = True       # Add evidence + explanation for why each report was generated

# BERTScore settings. None + lang="en" lets bert-score choose its standard English backbone.
# For a domain-specific experiment, try BERTSCORE_MODEL_TYPE="microsoft/BiomedVLP-CXR-BERT-specialized" and BERTSCORE_NUM_LAYERS=12.
# Custom model_type usually has no BERTScore baseline file, so Cell 9 will disable rescale automatically.
BERTSCORE_LANG = "en"
BERTSCORE_MODEL_TYPE = None
BERTSCORE_NUM_LAYERS = None
BERTSCORE_BATCH_SIZE = 8
BERTSCORE_RESCALE_WITH_BASELINE = True

# MedGemma 1.5 is gated on HF; accept terms and set HF_TOKEN if download fails.
MEDGEMMA_MODEL_ID = "google/medgemma-1.5-4b-it"
MEDGEMMA_SOURCE   = MEDGEMMA_ROOT or MEDGEMMA_MODEL_ID
REPORT_LANGUAGE   = "en"      # "en" keeps MIMIC metrics meaningful; set "vi" for Vietnamese reports

# Keep these aligned with the encoder topology used during training.
ENCODER_BIOVIL     = True
ENCODER_PUBMEDCLIP = True
ENCODER_SWIN       = False

print(f"EVAL_LIMIT={EVAL_LIMIT}  EVAL_SPLIT={EVAL_SPLIT}  BATCH_SIZE_IMG={BATCH_SIZE_IMG}  num_beams={NUM_BEAMS}")
print(f"MedGemma source: {MEDGEMMA_SOURCE}  report_language={REPORT_LANGUAGE}  explain_output={EXPLAIN_OUTPUT}")
print(f"BERTScore: run={RUN_BERTSCORE}, lang={BERTSCORE_LANG}, model={BERTSCORE_MODEL_TYPE or '(bert-score default)'}")
print(f"encoders: biovil={ENCODER_BIOVIL}, pubmedclip={ENCODER_PUBMEDCLIP}, swin={ENCODER_SWIN}")


In [ ]:
# Cell 6 — Write configs/env_config.yaml + eval_config.yaml for BLIP/MHCAC
import os
from pathlib import Path
from omegaconf import OmegaConf


def _resolve_or_empty(root, name):
    matches = list(Path(root).rglob(name))
    return str(matches[0]) if matches else ""


split_csv = PROCESSED_SPLIT_CSV or _resolve_or_empty(MIMIC_IMG_ROOT, "mimic-cxr-2.0.0-split.csv")
chexpert_csv = _resolve_or_empty(MIMIC_IMG_ROOT, "mimic-cxr-2.0.0-chexpert.csv")
metadata_csv = _resolve_or_empty(MIMIC_IMG_ROOT, "mimic-cxr-2.0.0-metadata.csv")
reports_csv = PROCESSED_ALL_CSV or PROCESSED_TEST_CSV

assert chexpert_csv, f"Missing mimic-cxr-2.0.0-chexpert.csv under {MIMIC_IMG_ROOT}"
assert metadata_csv, f"Missing mimic-cxr-2.0.0-metadata.csv under {MIMIC_IMG_ROOT}"

OUT_DIR = "/kaggle/working/eval_output"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("configs", exist_ok=True)

WANDB_PROJECT = "stage2_meta_cxr"

env_cfg = {
    "paths": {
        "data_root": "/kaggle/input",
        "mimic_cxr_jpg_root": MIMIC_IMG_ROOT,
        "split_csv": split_csv,
        "reports_csv": reports_csv,
        "chexpert_csv": chexpert_csv,
        "metadata_csv": metadata_csv,
        "processed_dir": PROCESSED_ROOT,
        "processed_train_csv": PROCESSED_TRAIN_CSV,
        "processed_val_csv": PROCESSED_VAL_CSV,
        "processed_test_csv": PROCESSED_TEST_CSV,
        "output_dir": OUT_DIR,
        "checkpoint_dir": OUT_DIR,
    },
    "wandb": {"entity": "", "project": WANDB_PROJECT},
    "java": {"home": JAVA_HOME, "path": JAVA_PATH},
}
with open("configs/env_config.yaml", "w") as f:
    OmegaConf.save(OmegaConf.create(env_cfg), f)
print("Wrote configs/env_config.yaml")
print(f"Evaluation split source: {EVAL_SPLIT} -> {env_cfg['paths'][f'processed_{EVAL_SPLIT}_csv']}")

# Load base inference config and override for eval
base_cfg = OmegaConf.load("pretraining/configs/blip2_pretrain_stage1_emb.yaml")
base_cfg.model.finetuned = QFORMER_CKPT
base_cfg.model.load_finetuned = True
base_cfg.model.encoders = OmegaConf.create({
    "biovil": bool(ENCODER_BIOVIL),
    "pubmedclip": bool(ENCODER_PUBMEDCLIP),
    "swin": bool(ENCODER_SWIN),
})
base_cfg.model.llm = OmegaConf.create({
    "provider": "medgemma",
    "model_id": MEDGEMMA_MODEL_ID,
    "source": MEDGEMMA_SOURCE,
})
base_cfg.run.evaluate = True
base_cfg.run.task = "image_text_pretrain_eval"
base_cfg.run.batch_size_eval = BATCH_SIZE_IMG
base_cfg.run.num_beams = NUM_BEAMS
base_cfg.run.max_len = MAX_NEW_TOKENS
base_cfg.run.output_dir = OUT_DIR
base_cfg.run.distributed = False
base_cfg.run.world_size = 1
# Remove resume_ckpt_path so runner doesn't try to load training state
if "resume_ckpt_path" in base_cfg.run:
    del base_cfg.run.resume_ckpt_path

EVAL_CFG_PATH = "configs/eval_config.yaml"
with open(EVAL_CFG_PATH, "w") as f:
    OmegaConf.save(base_cfg, f)
print(f"Wrote {EVAL_CFG_PATH}")
print(f"MedGemma will load from: {MEDGEMMA_SOURCE}")

# Initialize W&B run for the eval results to project `stage2_meta_cxr`.
WANDB_RUN = None
if HAVE_WANDB_CREDS:
    import time
    import wandb
    run_name = f"eval-{EVAL_SPLIT}-{time.strftime('%Y%m%d-%H%M%S')}"
    WANDB_RUN = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "eval_split": EVAL_SPLIT,
            "eval_limit": EVAL_LIMIT,
            "num_beams": NUM_BEAMS,
            "max_new_tokens": MAX_NEW_TOKENS,
            "batch_size_img": BATCH_SIZE_IMG,
            "medgemma_model_id": MEDGEMMA_MODEL_ID,
            "medgemma_source": MEDGEMMA_SOURCE,
            "report_language": REPORT_LANGUAGE,
            "encoders": {
                "biovil": bool(ENCODER_BIOVIL),
                "pubmedclip": bool(ENCODER_PUBMEDCLIP),
                "swin": bool(ENCODER_SWIN),
            },
            "qformer_ckpt": QFORMER_CKPT,
            "run_bertscore": RUN_BERTSCORE,
            "run_chexpert_f1": RUN_CHEXPERT_F1,
        },
        dir=OUT_DIR,
    )
    print(f"W&B run initialized: project={WANDB_PROJECT} run={run_name}")
else:
    print("W&B run skipped (no credentials).")


In [ ]:
# Cell 7 — Build Q-Former (BLIP/MHCAC) + MedGemma 1.5
# (init helpers copied from inference.py to avoid Gradio import side effects at module load.)
import os
import sys, argparse
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

# Patch sys.argv so the in-repo Config(parse_args()) sees our eval config path
sys.argv = ["eval_kaggle", "--cfg-path", EVAL_CFG_PATH]

from model.lavis.common.config import Config


def _parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--cfg-path", required=True)
    p.add_argument("--local_rank", type=int, default=0)
    p.add_argument("--options", nargs="+")
    return p.parse_args()


cfg = Config(_parse_args())

# Register all LAVIS components (must precede tasks.setup_task)
import model.lavis.tasks as tasks                  # noqa: E402
from model.lavis.datasets.builders import *        # noqa: E402, F401, F403
from model.lavis.models import *                   # noqa: E402, F401, F403
from model.lavis.processors import *               # noqa: E402, F401, F403
from model.lavis.runners import *                  # noqa: E402, F401, F403
from model.lavis.tasks import *                    # noqa: E402, F401, F403


def init_blip(cfg):
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    return model.to(torch.device("cpu"))


def _preferred_medgemma_dtype():
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


def init_medgemma(model_name_or_path):
    dtype = _preferred_medgemma_dtype()
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    hf_kwargs = {"token": hf_token} if hf_token else {}
    print(f"Loading MedGemma from {model_name_or_path} with dtype={dtype}...")
    processor = AutoProcessor.from_pretrained(model_name_or_path, **hf_kwargs)
    model = AutoModelForImageTextToText.from_pretrained(
        model_name_or_path,
        torch_dtype=dtype,
        device_map="auto",
        **hf_kwargs,
    )
    return model.eval(), processor, dtype


print("Building BLIP (Q-Former + MHCAC)...")
blip_model = init_blip(cfg).cuda().eval()

print("Building MedGemma 1.5...")
medgemma_model, medgemma_processor, MEDGEMMA_DTYPE = init_medgemma(MEDGEMMA_SOURCE)

# Sanity check: forward a dummy image through BLIP/MHCAC only.
with torch.no_grad():
    dummy = torch.randn(1, 3, 448, 448).cuda()
    out = blip_model.forward_image(dummy)
    logits = out[0]
    qf = out[1]
    print(f"BLIP forward_image OK -> logits {tuple(logits.shape)}  qformer_embs {tuple(qf.shape)}")
print("MedGemma ready.")


In [ ]:
# Cell 8 — Run inference on the eval split
import os, json, time
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset

ABN_NAMES = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other",
    "Fracture", "Support Devices",
]
CLASS_MAP = {"uncertain": 2, "positive": 1, "negative": 0}
CLASS_LABELS_VI = {
    "positive": "dương tính",
    "negative": "âm tính",
    "uncertain": "chưa chắc chắn",
    "below_threshold": "dưới ngưỡng",
}

ABN_NAMES_VI = {
    "No Finding": "không phát hiện bất thường",
    "Enlarged Cardiomediastinum": "bóng trung thất rộng",
    "Cardiomegaly": "tim to",
    "Lung Opacity": "đám mờ phổi",
    "Lung Lesion": "tổn thương phổi khu trú",
    "Edema": "phù phổi",
    "Consolidation": "đông đặc phổi",
    "Pneumonia": "viêm phổi",
    "Atelectasis": "xẹp phổi",
    "Pneumothorax": "tràn khí màng phổi",
    "Pleural Effusion": "tràn dịch màng phổi",
    "Pleural Other": "bất thường màng phổi khác",
    "Fracture": "gãy xương",
    "Support Devices": "dụng cụ hỗ trợ",
}


def _load_thresholds(thresholds_path):
    if thresholds_path and os.path.isfile(thresholds_path):
        with open(thresholds_path) as f:
            return json.load(f), True
    return {}, False


def build_abnormality_evidence(logits, thresholds_path=None, class_map=None):
    """Return prompt categories plus a full probability/decision table for explainability."""
    if class_map is None:
        class_map = CLASS_MAP
    if not isinstance(logits, torch.Tensor):
        logits = torch.tensor(logits)

    probabilities = torch.softmax(logits, dim=1).tolist()
    thresholds_data, use_thresholds = _load_thresholds(thresholds_path)
    categorized = {cls: [] for cls in class_map}
    evidence = []

    idx_to_class = {v: k for k, v in class_map.items()}
    for abn, probs in zip(ABN_NAMES, probabilities):
        thresholds_abn = thresholds_data.get(abn, {})
        thresholds = {cls: float(thresholds_abn.get(cls, 0.5)) for cls in class_map}

        if use_thresholds:
            decision, best_score = "below_threshold", 0.0
            for cls, cls_idx in class_map.items():
                prob = float(probs[cls_idx])
                if prob >= thresholds[cls] and prob > best_score:
                    decision, best_score = cls, prob
            if decision == "below_threshold":
                argmax_idx = int(torch.tensor(probs).argmax().item())
                selected_probability = float(probs[argmax_idx])
                argmax_class = idx_to_class[argmax_idx]
            else:
                selected_probability = best_score
                argmax_class = decision
        else:
            argmax_idx = int(torch.tensor(probs).argmax().item())
            decision = idx_to_class[argmax_idx]
            argmax_class = decision
            selected_probability = float(probs[argmax_idx])

        included_in_prompt = abn != "No Finding" and decision in categorized
        if included_in_prompt:
            categorized[decision].append(abn)

        evidence.append({
            "abnormality": abn,
            "abnormality_vi": ABN_NAMES_VI.get(abn, abn),
            "probabilities": {
                "negative": float(probs[class_map["negative"]]),
                "positive": float(probs[class_map["positive"]]),
                "uncertain": float(probs[class_map["uncertain"]]),
            },
            "thresholds": thresholds if use_thresholds else None,
            "decision": decision,
            "argmax_class": argmax_class,
            "selected_probability": selected_probability,
            "included_in_prompt": included_in_prompt,
        })

    return categorized, evidence, use_thresholds


def classify_abnormalities(logits, thresholds_path=None, class_map=None):
    categorized, _, _ = build_abnormality_evidence(logits, thresholds_path, class_map)
    return categorized


def _names_for_language(names):
    if REPORT_LANGUAGE.lower().startswith("vi"):
        return [ABN_NAMES_VI.get(name, name) for name in names]
    return names


def format_findings(cat):
    segs = []
    if REPORT_LANGUAGE.lower().startswith("vi"):
        if cat.get("positive"):
            segs.append("Bất thường dương tính: " + ", ".join(_names_for_language(cat["positive"])))
        if cat.get("negative"):
            segs.append("Bất thường âm tính: " + ", ".join(_names_for_language(cat["negative"])))
        if cat.get("uncertain"):
            segs.append("Bất thường chưa chắc chắn: " + ", ".join(_names_for_language(cat["uncertain"])))
        return ". ".join(segs) if segs else "không ghi nhận bất thường thường gặp"

    if cat.get("positive"):
        segs.append("Positive findings: " + ", ".join(cat["positive"]))
    if cat.get("negative"):
        segs.append("Negative findings: " + ", ".join(cat["negative"]))
    if cat.get("uncertain"):
        segs.append("Uncertain findings: " + ", ".join(cat["uncertain"]))
    return ". ".join(segs) if segs else "no common findings"


def _format_evidence_list(items, language):
    if not items:
        return "none" if language == "en" else "không có"
    chunks = []
    for item in items:
        name = item["abnormality"] if language == "en" else item.get("abnormality_vi", item["abnormality"])
        decision = item["decision"] if language == "en" else CLASS_LABELS_VI.get(item["decision"], item["decision"])
        chunks.append(f"{name} ({decision}, p={item['selected_probability']:.2f})")
    return "; ".join(chunks)


def build_report_explanation(findings_text, evidence, use_thresholds):
    """Deterministic explanation from model evidence, not an LLM self-rationale."""
    if not EXPLAIN_OUTPUT:
        return ""

    positives = [e for e in evidence if e["decision"] == "positive" and e["abnormality"] != "No Finding"]
    uncertains = [e for e in evidence if e["decision"] == "uncertain" and e["abnormality"] != "No Finding"]
    negatives = [e for e in evidence if e["decision"] == "negative" and e["abnormality"] != "No Finding"]
    below = [e for e in evidence if e["decision"] == "below_threshold" and e["abnormality"] != "No Finding"]
    threshold_note = "per-abnormality thresholds" if use_thresholds else "argmax probabilities"

    if REPORT_LANGUAGE.lower().startswith("vi"):
        return (
            "Báo cáo được sinh từ hai nguồn bằng chứng: ảnh X-quang đầu vào và tóm tắt bất thường có cấu trúc do BLIP/MHCAC dự đoán. "
            f"Tóm tắt đưa vào MedGemma là: '{findings_text}'. "
            f"Các nhãn dương tính chính: {_format_evidence_list(positives, 'vi')}. "
            f"Các nhãn chưa chắc chắn: {_format_evidence_list(uncertains, 'vi')}. "
            f"Các nhãn âm tính/dưới ngưỡng ({len(negatives)} âm tính, {len(below)} dưới ngưỡng) được dùng để giới hạn nội dung, tránh sinh thêm phát hiện không được hỗ trợ. "
            f"Quyết định nhãn dựa trên {'ngưỡng riêng từng bất thường' if use_thresholds else 'xác suất argmax'}; MedGemma chỉ được yêu cầu viết phần Findings từ các bằng chứng này."
        )

    return (
        "The report was generated from two evidence sources: the input chest X-ray and the structured abnormality summary predicted by BLIP/MHCAC. "
        f"The summary passed to MedGemma was: '{findings_text}'. "
        f"Main positive labels: {_format_evidence_list(positives, 'en')}. "
        f"Uncertain labels: {_format_evidence_list(uncertains, 'en')}. "
        f"Negative/below-threshold labels ({len(negatives)} negative, {len(below)} below threshold) constrain the report so unsupported findings are not added. "
        f"Label decisions used {threshold_note}; MedGemma was instructed to write only the Findings section from this evidence."
    )


THRESHOLD_PATH = cfg.config.model.mhcac.threshold_path

PROMPT_TAIL_EN = (
    "\n\nAct as an expert radiologist. Use the chest X-ray image and the structured abnormality information "
    "to write the Findings section of a chest X-ray report.\n\n"
    "- Do not invent findings that are not supported by the image or the structured abnormality information.\n"
    "- Do not repeat the same information using different wording.\n"
    "- Use a single, fluent paragraph in formal radiological style.\n"
    "- Use cautious and precise language if uncertain abnormalities are present.\n"
    "- Avoid enumeration, bullet points, and speculative phrases.\n\n"
    "Return only the generated findings text."
)

PROMPT_TAIL_VI = (
    "\n\nBạn là bác sĩ chẩn đoán hình ảnh. Hãy dùng ảnh X-quang ngực và thông tin bất thường có cấu trúc "
    "để viết phần Findings của báo cáo X-quang ngực bằng tiếng Việt chuyên ngành.\n\n"
    "- Không bịa thêm phát hiện nếu ảnh hoặc thông tin có cấu trúc không hỗ trợ.\n"
    "- Không lặp lại cùng một ý bằng nhiều cách diễn đạt khác nhau.\n"
    "- Viết một đoạn văn ngắn, mạch lạc, văn phong báo cáo X-quang.\n"
    "- Nếu có bất thường chưa chắc chắn, dùng ngôn ngữ thận trọng.\n"
    "- Không dùng gạch đầu dòng hoặc đánh số.\n\n"
    "Chỉ trả về nội dung Findings đã sinh."
)


def build_medgemma_messages(image, findings_text):
    if REPORT_LANGUAGE.lower().startswith("vi"):
        prompt = f"Thông tin bất thường có cấu trúc: {findings_text}{PROMPT_TAIL_VI}"
    else:
        prompt = f"Structured abnormality information: {findings_text}{PROMPT_TAIL_EN}"
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]


def generate_medgemma_report(image_path, findings_text):
    image = Image.open(image_path).convert("RGB")
    messages = build_medgemma_messages(image, findings_text)
    inputs = medgemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(medgemma_model.device, dtype=MEDGEMMA_DTYPE)

    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        generated = medgemma_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=False,
        )
    generated = generated[0][input_len:]
    return medgemma_processor.decode(generated, skip_special_tokens=True).strip()


dataset = MIMIC_CXR_Dataset(
    vis_processor=None, text_processor=None,
    vis_root=MIMIC_IMG_ROOT,
    split=EVAL_SPLIT, cfg=cfg, truncate=None,
)
print(f"{EVAL_SPLIT} split size: {len(dataset)}")

loader = DataLoader(
    dataset, batch_size=BATCH_SIZE_IMG, shuffle=False,
    num_workers=2, pin_memory=True,
)

PREDS_PATH = os.path.join(OUT_DIR, f"predictions_{EVAL_SPLIT}.json")
EXPLANATIONS_PATH = os.path.join(OUT_DIR, f"explanations_{EVAL_SPLIT}.json")
predictions, explanations, all_logits, all_labels = [], [], [], []
processed = 0
t0 = time.time()

with torch.no_grad():
    for batch in tqdm(loader, desc="Inference"):
        if EVAL_LIMIT is not None and processed >= EVAL_LIMIT:
            break
        images = batch["image"].cuda(non_blocking=True)
        out = blip_model.forward_image(images)
        logits = out[0]
        all_logits.append(logits.detach().cpu())
        all_labels.append(batch["classification_labels"])

        for j in range(images.size(0)):
            if EVAL_LIMIT is not None and processed >= EVAL_LIMIT:
                break
            dicom_id = batch["dicom_id"][j]
            image_path = batch["image_path"][j]
            cat, evidence, use_thresholds = build_abnormality_evidence(
                logits[j].detach().cpu(), thresholds_path=THRESHOLD_PATH
            )
            findings_text = format_findings(cat)
            pred_text = generate_medgemma_report(image_path, findings_text)
            explanation_text = build_report_explanation(findings_text, evidence, use_thresholds)

            record = {
                "image_id": int(dataset.img_ids[dicom_id]),
                "dicom_id": str(dicom_id),
                "caption": pred_text,
                "gt_findings": batch["text_output"][j] if "text_output" in batch else "",
                "findings_summary": findings_text,
                "explanation": explanation_text,
                "abnormality_evidence": evidence,
                "explanation_source": "BLIP/MHCAC class probabilities, thresholds, and the structured prompt passed to MedGemma",
                "llm": MEDGEMMA_MODEL_ID,
                "report_language": REPORT_LANGUAGE,
            }
            predictions.append(record)
            explanations.append({
                "image_id": record["image_id"],
                "dicom_id": record["dicom_id"],
                "caption": pred_text,
                "findings_summary": findings_text,
                "explanation": explanation_text,
                "abnormality_evidence": evidence,
            })
            processed += 1

            if processed % 100 == 0:
                with open(PREDS_PATH, "w") as f:
                    json.dump(predictions, f, ensure_ascii=False, indent=2)
                with open(EXPLANATIONS_PATH, "w") as f:
                    json.dump(explanations, f, ensure_ascii=False, indent=2)

with open(PREDS_PATH, "w") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
with open(EXPLANATIONS_PATH, "w") as f:
    json.dump(explanations, f, ensure_ascii=False, indent=2)
elapsed = time.time() - t0
print(f"Generated {len(predictions)} reports in {elapsed:.1f}s ({elapsed/max(len(predictions),1):.2f}s/sample)")
print(f"Predictions saved -> {PREDS_PATH}")
print(f"Explanations saved -> {EXPLANATIONS_PATH}")

# Save logits for CheXpert F1
ALL_LOGITS = torch.cat(all_logits, dim=0) if all_logits else torch.empty(0)
ALL_LABELS = torch.cat(all_labels, dim=0) if all_labels else torch.empty(0)
if EVAL_LIMIT is not None:
    ALL_LOGITS = ALL_LOGITS[:EVAL_LIMIT]
    ALL_LABELS = ALL_LABELS[:EVAL_LIMIT]
torch.save({"logits": ALL_LOGITS, "labels": ALL_LABELS}, os.path.join(OUT_DIR, "classification.pt"))
print(f"Saved classification tensor: logits {tuple(ALL_LOGITS.shape)}  labels {tuple(ALL_LABELS.shape)}")


In [ ]:
# Cell 9 — Compute BLEU-1/2/3/4 + METEOR + ROUGE-L + optional BERTScore
import os, json
import torch
from model.lavis.data.ReportDataset import MIMICEvalCap

gts_df = dataset.annotation[["dicom_id", "findings"]].copy()
evaluator = MIMICEvalCap(gts=gts_df, img_id_map=dataset.img_ids)
scores, gts_img_id = evaluator.evaluate(predictions)

if RUN_BERTSCORE and predictions:
    if not REPORT_LANGUAGE.lower().startswith("en"):
        print(
            "WARN: REPORT_LANGUAGE is not English while MIMIC-CXR references are English; "
            "BERTScore will not be directly comparable to English-report runs."
        )

    from bert_score import score as bertscore_score

    candidates = [str(p.get("caption", "")) for p in predictions]
    references = [str(p.get("gt_findings", "")) for p in predictions]
    rescale_with_baseline = BERTSCORE_RESCALE_WITH_BASELINE
    if BERTSCORE_MODEL_TYPE and BERTSCORE_RESCALE_WITH_BASELINE:
        print("WARN: custom BERTScore model_type selected; disabling rescale_with_baseline to avoid missing-baseline errors.")
        rescale_with_baseline = False

    bert_kwargs = {
        "lang": BERTSCORE_LANG,
        "batch_size": BERTSCORE_BATCH_SIZE,
        "rescale_with_baseline": rescale_with_baseline,
        "verbose": True,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }
    if BERTSCORE_MODEL_TYPE:
        bert_kwargs["model_type"] = BERTSCORE_MODEL_TYPE
    if BERTSCORE_NUM_LAYERS is not None:
        bert_kwargs["num_layers"] = BERTSCORE_NUM_LAYERS

    P, R, F1 = bertscore_score(candidates, references, **bert_kwargs)
    scores["BERTScore_P"] = float(P.mean().item())
    scores["BERTScore_R"] = float(R.mean().item())
    scores["BERTScore_F1"] = float(F1.mean().item())

    bertscore_rows = []
    for pred, p, r, f1 in zip(predictions, P.tolist(), R.tolist(), F1.tolist()):
        bertscore_rows.append({
            "image_id": int(pred["image_id"]),
            "dicom_id": str(pred["dicom_id"]),
            "BERTScore_P": float(p),
            "BERTScore_R": float(r),
            "BERTScore_F1": float(f1),
        })
    bertscore_path = os.path.join(OUT_DIR, f"bertscore_{EVAL_SPLIT}.json")
    with open(bertscore_path, "w") as f:
        json.dump(bertscore_rows, f, indent=2)
    print(f"Saved per-sample BERTScore -> {bertscore_path}")

print("\n=== Report-generation metrics ===")
for k in [
    "Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L",
    "BERTScore_P", "BERTScore_R", "BERTScore_F1", "agg_metrics",
]:
    if k in scores:
        print(f"  {k:14s}: {scores[k]:.4f}")

scores_path = os.path.join(OUT_DIR, "scores.json")
with open(scores_path, "w") as f:
    json.dump({k: float(v) for k, v in scores.items()}, f, indent=2)
print(f"Saved -> {scores_path}")


In [ ]:
# Cell 10 — CheXpert-style classification F1 over 14 abnormalities
import os, json
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

if RUN_CHEXPERT_F1 and ALL_LOGITS.numel() > 0:
    # Approach A: argmax over 3-way logits per abnormality, binary positive vs rest
    preds_cls  = ALL_LOGITS.argmax(dim=-1).numpy()    # (N, 14)
    labels_cls = ALL_LABELS.numpy()                    # (N, 14)
    assert preds_cls.shape == labels_cls.shape, f"Shape mismatch: {preds_cls.shape} vs {labels_cls.shape}"

    per_task = {}
    for t, name in enumerate(ABN_NAMES):
        y_true = (labels_cls[:, t] == 1).astype(int)
        y_pred = (preds_cls[:, t] == 1).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        per_task[name] = {
            "precision": float(p), "recall": float(r), "f1": float(f1), "acc": float(acc),
            "n_pos_true": int(y_true.sum()), "n_pos_pred": int(y_pred.sum()),
        }

    macro_p  = float(np.mean([v["precision"] for v in per_task.values()]))
    macro_r  = float(np.mean([v["recall"]    for v in per_task.values()]))
    macro_f1 = float(np.mean([v["f1"]        for v in per_task.values()]))

    print("\n=== CheXpert classification F1 (positive class) ===")
    header = f"{'Abnormality':30s}  Prec   Recall  F1     n_pos_true  n_pos_pred"
    print(header)
    print("-" * len(header))
    for name in ABN_NAMES:
        v = per_task[name]
        print(f"{name:30s}  {v['precision']:.3f}  {v['recall']:.3f}  {v['f1']:.3f}"
              f"  {v['n_pos_true']:5d}       {v['n_pos_pred']:5d}")
    print(f"\nMacro avg -> Prec: {macro_p:.4f}  Recall: {macro_r:.4f}  F1: {macro_f1:.4f}")

    chex_path = os.path.join(OUT_DIR, "chexpert_f1.json")
    with open(chex_path, "w") as f:
        json.dump({
            "per_task": per_task,
            "macro": {"precision": macro_p, "recall": macro_r, "f1": macro_f1},
        }, f, indent=2)
    print(f"Saved -> {chex_path}")
else:
    print("Skipped CheXpert F1 (RUN_CHEXPERT_F1=False or no logits collected)")

# Approach B - Java CheXpert labeler on generated reports (optional, fragile)
if TRY_TEXT_LABELER:
    print("\nWARN: Java-based CheXpert text labeler is NOT bundled in this notebook.")
    print("  To enable: attach a chexpert-labeler Kaggle dataset (or clone https://github.com/stanfordmlgroup/chexpert-labeler)")
    print("  then implement subprocess invocation here. Approach A above is the primary metric.")


In [ ]:
# Cell 10b — Log evaluation results to W&B project `stage2_meta_cxr`
import os, json

if WANDB_RUN is not None:
    import wandb

    # Log scalar metrics
    metric_payload = {f"eval/{k}": float(v) for k, v in scores.items()}
    if RUN_CHEXPERT_F1 and ALL_LOGITS.numel() > 0:
        metric_payload["chexpert/macro_precision"] = macro_p
        metric_payload["chexpert/macro_recall"] = macro_r
        metric_payload["chexpert/macro_f1"] = macro_f1
        for abn_name, stats in per_task.items():
            tag = abn_name.replace(" ", "_")
            metric_payload[f"chexpert/per_task/{tag}/f1"] = stats["f1"]
            metric_payload[f"chexpert/per_task/{tag}/precision"] = stats["precision"]
            metric_payload[f"chexpert/per_task/{tag}/recall"] = stats["recall"]
    metric_payload["eval/num_samples"] = len(predictions)
    wandb.log(metric_payload)

    # Log a small predictions table (cap to avoid huge uploads)
    PREVIEW_ROWS = min(len(predictions), 200)
    table = wandb.Table(columns=["image_id", "dicom_id", "gt_findings", "generated", "findings_summary"])
    for rec in predictions[:PREVIEW_ROWS]:
        table.add_data(
            rec.get("image_id"),
            rec.get("dicom_id", ""),
            rec.get("gt_findings", ""),
            rec.get("caption", ""),
            rec.get("findings_summary", ""),
        )
    wandb.log({f"predictions_preview_{EVAL_SPLIT}": table})

    # Upload result files as a single artifact
    artifact = wandb.Artifact(
        name=f"eval_results_{EVAL_SPLIT}",
        type="evaluation",
        metadata={"split": EVAL_SPLIT, "num_samples": len(predictions)},
    )
    for fname in [
        f"predictions_{EVAL_SPLIT}.json",
        f"explanations_{EVAL_SPLIT}.json",
        f"bertscore_{EVAL_SPLIT}.json",
        "scores.json",
        "chexpert_f1.json",
        "classification.pt",
    ]:
        src = os.path.join(OUT_DIR, fname)
        if os.path.exists(src):
            artifact.add_file(src)
    wandb.log_artifact(artifact)

    wandb.finish()
    print(f"Logged metrics + artifact to W&B project `{WANDB_PROJECT}`.")
else:
    print("Skipped W&B logging (no run was initialized; set WANDB_API_KEY in Kaggle Secrets).")


In [ ]:
# Cell 11 — (Optional) Push evaluation outputs back to a Kaggle dataset version
import os, json, shutil, subprocess, time

if HAVE_KAGGLE_CREDS:
    PUSH_DIR = "/kaggle/working/eval_results_push"
    os.makedirs(PUSH_DIR, exist_ok=True)

    result_files = [
        f"predictions_{EVAL_SPLIT}.json",
        f"explanations_{EVAL_SPLIT}.json",
        f"bertscore_{EVAL_SPLIT}.json",
        "scores.json",
        "chexpert_f1.json",
        "classification.pt",
    ]
    for fname in result_files:
        src = os.path.join(OUT_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, PUSH_DIR)

    DATASET_SLUG = f"{KAGGLE_USER}/meta-cxr-eval-results"
    meta = {
        "title": "META-CXR evaluation outputs",
        "id": DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(PUSH_DIR, "dataset-metadata.json"), "w") as f:
        json.dump(meta, f)

    timestamp = time.strftime("%Y-%m-%d %H:%M")
    try:
        subprocess.check_call(
            ["kaggle", "datasets", "version", "-p", PUSH_DIR, "-m", f"Eval run {timestamp}"]
        )
        print(f"OK - Pushed new version of {DATASET_SLUG}")
    except subprocess.CalledProcessError:
        try:
            subprocess.check_call(["kaggle", "datasets", "create", "-p", PUSH_DIR])
            print(f"OK - Created new dataset {DATASET_SLUG}")
        except subprocess.CalledProcessError as e:
            print(f"WARN - Push failed: {e}. Files remain at {PUSH_DIR}.")
else:
    print("Skipped Kaggle push (no credentials configured in Cell 2).")
